In [ ]:
# ParDo interaction with DoFn (In this case it's used to count the number of character per element in a list of strings)
from dataclasses import dataclass

import apache_beam as beam

# The input PCollection of Strings.
input = beam.Create(["apple", "banana", "cherry", "durian", "guava", "melon"])


# @dataclass(frozen=True)
@dataclass
class String(): 
    value: str

    def __len__(self):
        return len(self.value)

# The DoFn to perform on each element in the input PCollection.
class ComputeWordLengthFn(beam.DoFn):
    # NOTE: Beam does not guarantee the number of invocations for `process`, the number may change due to different factors
    # user implementation should not rely on the number of invocations directly (i.e. counter) or indirectly (i.e. stateful code)
    # NOTE: Immutability requirements:
    #   1. You should not modify the arguments or any side inputs passes to `process`.
    def process(self, element):
        # NOTE: 
        # Q: Why do we need to yield or return a list of items, what happens when we return the raw element?
        # A: Beam expects `process` to return an iterable, `int` is not an iterable.
        # Q: Does that mean the by design beam's `DoFn` are meant to process one-to-many relationships?
        # A: No, According to "tour of Beam" "ParDo" emits 0, 1 or many elements in the form of `PCollection`

        # NOTE: Beam's SDK does not guard against mutating the input or side-input.
        #       The immutability requirement is considered a "programming contract" not something the Python SDK
        #       actively guards against. To help mitigate potential risks we can use `@dataclass(frozen=True)` when possible.
        # element = String("Hello, world!")
        # element.value = "Hi!"

        yield len(element)          # this works
        # return [len(element)]       # this also works
        # return len(element)           # this does not work. Fails with the error: "ERROR:apache_beam.runners.common:'int' object is not iterable"


with beam.Pipeline() as p:
    # Apply a ParDo to the PCollection [input] to compute lengths for each word.
    (p 
    | input 
    | beam.Map(String)
    | beam.ParDo(ComputeWordLengthFn()) # only added to verify if there's any guards against process input mutability.
    | beam.Map(print)
    )


5
6
6
6
5
5


In [ ]:
#   Licensed to the Apache Software Foundation (ASF) under one
#   or more contributor license agreements.  See the NOTICE file
#   distributed with this work for additional information
#   regarding copyright ownership.  The ASF licenses this file
#   to you under the Apache License, Version 2.0 (the
#   "License"); you may not use this file except in compliance
#   with the License.  You may obtain a copy of the License at
#
#       http://www.apache.org/licenses/LICENSE-2.0
#
#   Unless required by applicable law or agreed to in writing, software
#   distributed under the License is distributed on an "AS IS" BASIS,
#   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#   See the License for the specific language governing permissions and
#   limitations under the License.


import apache_beam as beam

# Output PCollection
class Output(beam.PTransform):
    class _OutputFn(beam.DoFn):
        def __init__(self, prefix=''):
            super().__init__()
            self.prefix = prefix

        def process(self, element):
            print(self.prefix+str(element))

    def __init__(self, label=None,prefix=''):
        super().__init__(label)
        self.prefix = prefix

    def expand(self, input):
        input | beam.ParDo(self._OutputFn(self.prefix))

# Multiplications by 10
class MultiplyByTenDoFn(beam.DoFn):
    def process(self, element):
        yield element * 10


with beam.Pipeline() as p:
  (p | beam.Create([1, 2, 3, 4, 5])
    # Transform simple DoFn operation
     | beam.ParDo(MultiplyByTenDoFn())
     | Output())